# Data Collection

## 1. `pybaseball` And `baseballr`

Getting data directly from websites like [Baseball Reference](https://www.baseball-reference.com/) is challenging because such sites are *not designed for automated, large-scale extraction*. Their pages rely on complex and often inconsistent HTML structures that can change without notice, particularly during the season, making scraping brittle and unreliable. In addition, rate limits and other anti-scraping protections are used to manage server load and enforce terms of service, meaning repeated automated requests may result in blocked IP addresses or incomplete data retrieval. Relevant statistics are also spread across hundreds of player, team, and season pages rather than being provided as unified, downloadable datasets, making manual collection or basic scraping time-consuming and error-prone.  

As a result, most baseball analysts rely on specialized data libraries such as `pybaseball` and `baseballr`, which provide stable, structured, and terms-compliant access to baseball data without scraping websites directly. These libraries function *similarly to APIs* by offering programmatic access to baseball statistics through a combination of real APIs, such as MLB's Stats API and Baseball Savant's Statcast data endpoints, and well-structured public data sources, supplemented by reliable HTML parsing when necessary. The `pybaseball` library, a Python package, acts as a wrapper around multiple public baseball data sources including [FanGraphs](https://www.fangraphs.com/), Baseball Reference, [Baseball Savant](https://baseballsavant.mlb.com/), and [Retrosheet](https://www.retrosheet.org/), allowing users to *download statistics directly into Python* as Pandas `DataFrame` objects for analysis. With functions for querying basic and advanced batting metrics, Statcast data, schedules, standings, and more, `pybaseball` streamlines the data-collection process and allows researchers to focus on modeling and visualization rather than data acquisition.  

The `baseballr` library provides comparable functionality within the *R environment*, returning baseball data as tibbles that integrate naturally with the `tidyverse` ecosystem, including packages such as `dplyr`, `ggplot2`, and `tidymodels`. While both libraries offer robust tools for baseball analytics, this project uses `pybaseball` due to greater familiarity with Python and its data science libraries.  

[@pybaseball; @baseballr]

## 2. Statcast

## 2. Exploring `pybaseball` 

### A. Pulling Basic Stats (By Year)

#### i. Batting

In [ ]:
from pybaseball import batting_stats

In [ ]:
# 2025
df_bat = batting_stats(2025, qual = 0)
df_bat.head()

Note: This request gathers the batting/hitting stats for every MLB player who recorded a plate appearance in 2025. The `DataFrame` output includes *all types* of metrics. The same is true for the following two pulls. Please note that I will be using the terms *"data," "stats," and "metrics"* interchangeably throughout this project.

#### ii. Pitching

In [ ]:
from pybaseball import pitching_stats

In [ ]:
# 2025
df_pitch = pitching_stats(2025, qual = 0)
df_pitch.head()

#### iii. Fielding

In [ ]:
from pybaseball import fielding_stats

In [ ]:
# 2025
df_field = fielding_stats(2025, qual = 0)
df_field.head()

#### iv. Standings

In [ ]:
from pybaseball import standings

In [ ]:
# 2025
stand = standings(2025)
print(stand, "\n")

Note: The output here is a *list*, not a Pandas `DataFrame`!

### B. Pulling Statcast Data (By Date Range)

#### i. All Players

In [ ]:
from pybaseball import statcast

In [ ]:
# Month of April, 2025!
df_league_date = statcast(start_dt = "2025-04-01", end_dt = "2025-04-30")
df_league_date.head()

Note: This extraction, using the `statcast` function, gets *pitch-level data*, hence why there is a warning when the code is executed. Statcast measures pretty much everything down to the angle at which a pitch is released, for example. Therefore, I use a different function, namely `statcast_batter_exitvelo_barrels`, that retrieves aggregated data (by season) below. It is true, though, that pulling from FanGraphs using `batting_stats` above gets one the necessary Statcast data, hence why I remove some duplicate columns in `Data_Cleaning.ipynb`; I pull from both FanGraphs AND Statcast.

#### ii. One Player

In [ ]:
from pybaseball import playerid_lookup, statcast_batter

In [ ]:
# Find the appropriate ID!
judge_id = playerid_lookup("Judge", "Aaron", fuzzy = False)
print("Aaron Judge IDs:", judge_id.to_dict(orient = "records"), "\n")

# Pull Statcast data for Aaron Judge, April, 2025!
df_judge = statcast_batter("2025-04-01", "2025-04-30", 592450)
df_judge.head()

Note: Notice how there are *different IDs for FanGraphs and Statcast (MLBAM)*. This becomes important below!

### C. Pulling Team-Level Data

#### i. Schedule And Record

In [ ]:
from pybaseball import schedule_and_record

In [ ]:
# 2025 Yankees
df_nyy = schedule_and_record(2025, "NYY")
df_nyy.head()

#### ii. Batting

In [ ]:
from pybaseball import team_batting

In [ ]:
# 2025 
df_bat_team = team_batting(2025)
df_bat_team.head()

Note: When one pulls team-level data using `team_batting` as seen here, the *season totals* for counting stats such as home runs, for example, are what's outputted. For percentage stats such as OPS, the output in the `DataFrame` is a team average. The same is true for the following two examples.

#### iii. Pitching

In [ ]:
from pybaseball import team_pitching

In [ ]:
# 2025
df_pitch_team = team_pitching(2025)
df_pitch_team.head()

#### iv. Fielding

In [ ]:
from pybaseball import team_fielding

In [ ]:
# 2025
df_field_team = team_fielding(2025)
df_field_team.head()

## 3. Collecting FanGraphs Data

### A. Necessary Packages (More)

In [ ]:
import pandas as pd
from pybaseball import cache

### B. Setting Up The Pull

In [ ]:
# Enable local caching so repeated queries do not keep hitting remote sources!
cache.enable()

# Set project configuration!
START_SEASON = 2017
END_SEASON = 2025
MIN_PA = 200 # Minimum plate appearances to include a player in a season!
SEASONS = list(range(START_SEASON, END_SEASON + 1))

### C. Getting Standard Batting Stats (2017-2025), Adjusting `season` Column (For Later)

In [ ]:
# Pull FanGraphs stats for all MLB hitters from `START_SEASON` to `END_SEASON`!
all_seasons = []
for season in SEASONS:
    season_df = batting_stats(season, qual = 0) # `qual = 0` to get all players, 
                                                # regardless of plate appearances (FanGraphs does filtering of their own)!
    
    # Filter for hitters with at least `MIN_PA` plate appearances!
    season_df = season_df[season_df["PA"] >= MIN_PA].copy()
    
    """
    Overwrite FanGraphs' `Season` column with the standardized version!
    Ensure only one clean `season` column exists!
    """
    season_df = season_df.rename(columns = {"Season": "season"})
    season_df["season"] = season
    all_seasons.append(season_df)

# Concatenate all seasons into one `DataFrame`!
standard_raw_all = pd.concat(all_seasons, ignore_index = True)
print(f"Shape of dataset: {standard_raw_all.shape}\n")
standard_raw_all.head()

### D. Finding Aaron Judge's Player IDs, Customizing The Player Identifier 

In [ ]:
# Look up Aaron Judge (same as above)!
judge_ids = playerid_lookup("Judge", "Aaron", fuzzy = False)
print("Aaron Judge IDs again:", judge_ids.to_dict(orient = "records"), "\n")

# Standardize the `IDfg` column name!
standard_raw_all = standard_raw_all.rename(columns = {"IDfg": "fangraphs_id"})

# Ensure FanGraphs ID is numeric!
standard_raw_all["fangraphs_id"] = pd.to_numeric(standard_raw_all["fangraphs_id"], errors = "coerce")

# Check for missing FanGraphs IDs!
missing_fg = standard_raw_all["fangraphs_id"].isna().sum()
print(f"Missing FanGraphs IDs: {missing_fg}\n")

# Display potential duplicates by name and season!
duplicates = standard_raw_all[standard_raw_all.duplicated(subset = ["Name", "season"], keep = False)]
print(f"Number of potential duplicate name-season pairs: {duplicates.shape[0]}\n")
duplicates.head()

Note: I address *duplicates* more below!

### E. Getting Aaron Judge's Batting Stats, Saving `standard_raw_all` Initially

In [ ]:
# Extract Aaron Judge's rows using his FanGraphs ID!
judge_id_fg = 15640
judge_raw = standard_raw_all[standard_raw_all["fangraphs_id"] == judge_id_fg].copy()
print("Aaron Judge rows found:\n")
print(judge_raw[["Name", "season", "PA", "HR", "wRC+"]])

# Save the raw combined dataset only!
standard_raw_all.to_csv("../data/raw/standard_raw_all.csv", index = False)

Note: 

- The 2020 season was shortened due to the *COVID-19 pandemic*, so Judge had $< 200$ plate appearances that year. Therefore, I will exclude 2020 from the analysis to ensure consistency. I address this more below.

- I am only isolating Judge's stats here as a sanity check and will save them separately later.

### F. Adjusting `standard_raw_all` More

In [ ]:
# Check for true duplicates by `fangraphs_id` and `season` (should generally be zero)!
dup_id_season = standard_raw_all[standard_raw_all.duplicated(subset = ["fangraphs_id", "season"], keep = False)]
print(f"Number of true duplicates: {dup_id_season.shape[0]}\n")

# Drop the 2020 season for all hitters!
standard_raw_all = standard_raw_all[standard_raw_all["season"] != 2020].copy()
print(f"Shape after dropping 2020: {standard_raw_all.shape}\n")

# Save the updated raw dataset (with 2020 excluded)!
standard_raw_all.to_csv("../data/raw/standard_raw_all.csv", index = False)

## 4. Collecting Statcast Data

### A. Mapping IDs Between FanGraphs And Statcast 

In [ ]:
from pybaseball import playerid_reverse_lookup

In [ ]:
# Collect unique FanGraphs IDs from the raw batting data!
unique_fg_ids = standard_raw_all["fangraphs_id"].dropna().astype(int).unique().tolist()

# Look up cross-IDs for all of these FanGraphs IDs!
id_map = playerid_reverse_lookup(unique_fg_ids, key_type = "fangraphs")

# Keep and rename the columns!
id_map = id_map.rename(columns = {"key_fangraphs": "fangraphs_id",
                                  "key_mlbam": "mlbam_id"})
id_map = id_map[["fangraphs_id", "mlbam_id"]]
print("IDs:\n")
id_map.head()

### B. Merging IDs With `standard_raw_all`

In [ ]:
# Left join the ID mapping back to the main dataset!
batting_with_ids = standard_raw_all.merge(id_map, on = "fangraphs_id", how = "left")

# Check for missing MLBAM IDs after the merge!
missing_mlbam = batting_with_ids["mlbam_id"].isna().sum()
print(f"Missing MLBAM IDs: {missing_mlbam}\n")
batting_with_ids.head()

### C. Dropping Players Without MLBAM IDs

In [ ]:
# Remove players with no MLBAM ID, since they have no Statcast data!
before = batting_with_ids.shape[0]
batting_with_ids = batting_with_ids.dropna(subset = ["mlbam_id"]).copy()
batting_with_ids["mlbam_id"] = batting_with_ids["mlbam_id"].astype(int)
after = batting_with_ids.shape[0]

# Report how many players were removed due to missing MLBAM IDs!
print(f"Removed {before - after} players with missing MLBAM IDs.\n")
print(f"Remaining rows: {after}\n")
batting_with_ids.head()

### D. Pulling Statcast Batting Data (2017-2025)

In [ ]:
from pybaseball import statcast_batter_exitvelo_barrels

In [ ]:
# Pull Statcast aggregates for all hitters (excluding 2020)!
statcast_list = []
for season in SEASONS:
    if season == 2020:
        continue # Skip COVID season!
    df = statcast_batter_exitvelo_barrels(season)
    df["season"] = season
    statcast_list.append(df)

# Concatenate all seasons into one `DataFrame`!
statcast_raw_all = pd.concat(statcast_list, ignore_index = True)
print(f"Statcast raw shape: {statcast_raw_all.shape}\n")
statcast_raw_all.head()

Note: The Statcast dataset is *smaller (in terms of rows)* than the FanGraphs batting dataset because Statcast only includes players who generated measurable batted-ball events in a given season. While FanGraphs reports statistics for every hitter who recorded plate appearances, Statcast requires actual tracked contact, such as exit velocity or launch angle, to generate a profile. As a result, players with limited playing time, few balls in play, injury-shortened seasons, or data inconsistencies may be present in FanGraphs but missing from Statcast. This discrepancy is expected and does not affect the analysis, since all meaningful comparison hitters, including Aaron Judge, have complete Statcast data.

### E. Saving FanGraphs Data

In [ ]:
# Extract Aaron Judge's batting data!
judge_mlbam_id = 592450 
standard_raw_judge = batting_with_ids[batting_with_ids["mlbam_id"] == judge_mlbam_id].copy()
print("Judge batting rows found:\n")
print(standard_raw_judge[["Name", "season", "PA", "HR", "wRC+"]])

# Save FINAL raw datasets!
batting_with_ids.to_csv("../data/raw/standard_raw_all.csv", index = False)
standard_raw_judge.to_csv("../data/raw/standard_raw_judge.csv", index = False)

### F. Saving Statcast Data

In [ ]:
# Keep only Statcast rows for hitters who appear in the batting dataset!
valid_mlbam_ids = batting_with_ids["mlbam_id"].unique().tolist()
statcast_raw_all = statcast_raw_all[statcast_raw_all["player_id"].isin(valid_mlbam_ids)].copy()
print(f"Statcast rows after filtering to hitters with batting data: {statcast_raw_all.shape[0]}\n")

# Extract Aaron Judge's Statcast data using his MLBAM ID!
judge_mlbam_id = 592450
statcast_raw_judge = statcast_raw_all[statcast_raw_all["player_id"] == judge_mlbam_id].copy()
print("Some Judge Statcast rows found:\n")
print(statcast_raw_judge.head())

# Save the full and Judge-only Statcast datasets!
statcast_raw_all.to_csv("../data/raw/statcast_raw_all.csv", index = False)
statcast_raw_judge.to_csv("../data/raw/statcast_raw_judge.csv", index = False)